# Fine-tuning ModernBERT-base on Banking77

**Task**: 77-way intent classification on banking customer queries.

**Recipe**: full fine-tune (not LoRA) with `transformers.Trainer`, bf16 mixed precision, cosine LR schedule with 10% warmup, label-smoothing 0.1, 4 epochs, batch size 32.

**Why this recipe?** ModernBERT-base is 149M params — small enough to fully fine-tune on a free Colab T4. Banking77 has only 13k training examples, so full fine-tuning beats LoRA at this scale. Label smoothing pairs with temperature scaling in `03_error_analysis.ipynb` to give well-calibrated confidence outputs.

**Time**: ~15 min on a T4.

In [ ]:
# Colab: uncomment to install
# !pip install -q 'transformers>=4.48' 'datasets>=3.1' accelerate scikit-learn evaluate pydantic pydantic-settings

In [ ]:
import sys
sys.path.insert(0, '..')
from src.data import load_banking77_splits, label_names
splits = load_banking77_splits()
names = label_names()
print(splits)
print(f'num_labels = {len(names)}')
print(f'example: {splits["train"][0]}')

## The data

Banking77 has a `train`/`test` split out of the box. I carve a 10% stratified-by-label validation slice out of `train` so each of the 77 classes is represented. **The test set is held out and only touched once at the end** — touching it earlier for tuning decisions invalidates the final score.

In [ ]:
# Class balance check — Banking77 isn't perfectly balanced; we want to know.
import pandas as pd
counts = pd.Series(splits['train']['label']).value_counts()
print(f'min class size: {counts.min()}, max: {counts.max()}, median: {counts.median()}')

## Load the model

`AutoModelForSequenceClassification` wraps the base encoder + a fresh linear head sized for our 77 labels. The base weights are loaded; the head is randomly initialized and trained from scratch.

In [ ]:
from src.model import load_for_training
id2label = dict(enumerate(names))
model, tokenizer = load_for_training(num_labels=len(names), id2label=id2label)
print(f'trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## Tokenize

In [ ]:
from src.data import tokenize_dataset
train_tok = tokenize_dataset(splits['train'], tokenizer)
val_tok = tokenize_dataset(splits['validation'], tokenizer)
test_tok = tokenize_dataset(splits['test'], tokenizer)
print(train_tok)

## Train

All knobs live in `src/config.py` so the same recipe is reproducible from the CLI (`python -m src.train`) and the notebook. We select the best checkpoint by macro F1 on the validation set.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments
from src.config import settings

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

args = TrainingArguments(
    output_dir='./outputs/modernbert-banking77',
    num_train_epochs=settings.num_epochs,
    per_device_train_batch_size=settings.per_device_batch_size,
    per_device_eval_batch_size=settings.per_device_batch_size,
    learning_rate=settings.learning_rate,
    weight_decay=settings.weight_decay,
    warmup_ratio=settings.warmup_ratio,
    lr_scheduler_type='cosine',
    label_smoothing_factor=settings.label_smoothing,
    bf16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=25,
    report_to='none',
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

**What you should see:**
- Epoch 1: macro F1 climbs from random (~1.3%) to ~85%.
- Epoch 2-3: improvements taper to ~92%.
- Epoch 4: small gains; best checkpoint usually epoch 3 or 4.
- If val F1 starts dropping while train loss falls, lower `num_epochs` to 3.

In [ ]:
trainer.save_model('./outputs/modernbert-banking77')
tokenizer.save_pretrained('./outputs/modernbert-banking77')
print('Model saved.')

## Final test eval + calibration

Run `python -m src.eval --model_dir ./outputs/modernbert-banking77 --output ./outputs/eval_results.json` from a terminal cell, or run the cells in `03_error_analysis.ipynb` to inspect the confusion matrix and per-class breakdown.

## Push to the Hub

```bash
python scripts/push_to_hub.py \
    --model_dir ./outputs/modernbert-banking77 \
    --repo_id <your-username>/modernbert-banking77
```

The script generates a model card with the training recipe and uploads everything in one shot.